In [1]:
pip install pandas requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import sqlite3
import requests
import random
from datetime import datetime

def fetch_live_fx_rates():
    """Fetches real-time baseline exchange rates from the Frankfurter API."""
    print("Fetching live market rates from Frankfurter API...")
    url = "https://api.frankfurter.app/latest?from=USD"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return data['rates']
    except requests.exceptions.RequestException as e:
        print(f"API Error: {e}")
        return None

def generate_mock_transactions():
    """Generates synthetic cross-border transaction data for testing."""
    currencies = ['EUR', 'GBP', 'PHP', 'JPY', 'AUD']
    data = []
    
    for i in range(1, 501): # Generate 500 transactions
        target_currency = random.choice(currencies)
        amount_usd = round(random.uniform(500, 15000), 2)
        
        # Simulate the rate a bad payment provider gave the business (worse than market)
        provider_markup = random.uniform(1.01, 1.04) # 1% to 4% hidden markup
        
        data.append({
            'Transaction_ID': f"TXN-{1000 + i}",
            'Date': datetime.today().strftime('%Y-%m-%d'),
            'Base_Currency': 'USD',
            'Target_Currency': target_currency,
            'Transfer_Amount_USD': amount_usd,
            'Provider_Markup_Factor': provider_markup,
            'Status': random.choices(['Completed', 'Delayed', 'Failed'], weights=[85, 10, 5])[0]
        })
    return pd.DataFrame(data)

def process_and_flag_data(df, market_rates):
    """Applies operational finance and compliance business logic."""
    print("Processing financial data and applying compliance rules...")
    
    # Map the real market rate to the dataset
    df['Real_Market_Rate'] = df['Target_Currency'].map(market_rates)
    
    # Calculate what the provider actually charged
    df['Applied_Provider_Rate'] = df['Real_Market_Rate'] * df['Provider_Markup_Factor']
    
    # Calculate the Cost Leakage (Money lost to bad spreads)
    df['Cost_Leakage_USD'] = (df['Applied_Provider_Rate'] - df['Real_Market_Rate']) / df['Real_Market_Rate'] * df['Transfer_Amount_USD']
    
    # Apply Compliance & Risk Logic
    df['Compliance_Flag'] = 'CLEARED'
    
    # Flag 1: High Value Transaction
    df.loc[df['Transfer_Amount_USD'] > 10000, 'Compliance_Flag'] = 'REVIEW: HIGH VALUE'
    
    # Flag 2: Excessive FX Spread (> 3%)
    df.loc[df['Provider_Markup_Factor'] > 1.03, 'Compliance_Flag'] = 'REVIEW: HIGH SPREAD'
    
    # Clean up columns for presentation
    df = df.round(2)
    return df

def load_to_sql(df):
    """Loads the processed data into a local SQLite database for Power BI."""
    db_name = 'fx_operations.db'
    print(f"Loading data into SQL database: {db_name}...")
    
    # Connect to SQLite (this creates the file if it doesn't exist)
    conn = sqlite3.connect(db_name)
    
    # Write the data to a SQL table named 'Transactions'
    df.to_sql('Transactions', conn, if_exists='replace', index=False)
    conn.close()
    print("Pipeline execution complete! Data is ready for Power BI.")

# --- Execute the Pipeline ---
if __name__ == "__main__":
    market_rates = fetch_live_fx_rates()
    
    if market_rates:
        raw_df = generate_mock_transactions()
        processed_df = process_and_flag_data(raw_df, market_rates)
        load_to_sql(processed_df)

Fetching live market rates from Frankfurter API...
Processing financial data and applying compliance rules...
Loading data into SQL database: fx_operations.db...
Pipeline execution complete! Data is ready for Power BI.


In [7]:
# --- Execute the Pipeline ---
if __name__ == "__main__":
    market_rates = fetch_live_fx_rates()
    
    if market_rates:
        raw_df = generate_mock_transactions()
        processed_df = process_and_flag_data(raw_df, market_rates)
        
        # Load to SQL Database
        load_to_sql(processed_df)
        
        # Generate the CSV for Power BI
        processed_df.to_csv('fx_operations_clean.csv', index=False)
        print("CSV created successfully! Ready for Power BI.")

Fetching live market rates from Frankfurter API...
Processing financial data and applying compliance rules...
Loading data into SQL database: fx_operations.db...
Pipeline execution complete! Data is ready for Power BI.
CSV created successfully! Ready for Power BI.
